In [ ]:
import gc
import os
import time
import random
import pickle
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sys import getsizeof
import tensorflow as tf
print(tf.__version__)
from tensorflow.keras import metrics
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import layers, callbacks
from tensorflow.keras.models import Sequential, Model
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from tensorflow.keras import layers, models
from sklearn.preprocessing import LabelEncoder

! pip install tensorflow

2.20.0


In [ ]:
from google.colab import drive  #Jesli nie używasz google drive to usun datasourcePath przy load
drive.mount('/content/drive')

dataSourcePath = '/content/drive/MyDrive/BIAI/'

Mounted at /content/drive


In [ ]:
X = np.load(dataSourcePath+"Dane32PrzetworzoneNOAVGNOCLIP.npy")
y_txt = np.load(dataSourcePath+"EtykietyDanych.npy", allow_pickle=True)

X = np.transpose(X, (0,2,3,1))
# Wymiary pojedynczego obrazu
input_shape = X.shape[1:]


In [ ]:
print(X.shape)
print(y_txt.shape)
print(input_shape)

(1267, 34, 421, 21)
(1267,)
(34, 421, 21)


In [ ]:
labelEnc = LabelEncoder()
y = labelEnc.fit_transform(y_txt)
l_klas = len(labelEnc.classes_)
print(labelEnc.classes_)

['abstract' 'airplane' 'apple' 'banana' 'bird' 'boat' 'car' 'dog' 'person'
 'train' 'zebra']


In [ ]:
# PODZIAŁ 40-40-20 - pies
#60-20-20 - jabłko
#72-18-10 - ptak
X_train_validate, X_test, y_train_validate, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_validate, y_train, y_validate = train_test_split(X_train_validate, y_train_validate, test_size=0.25, random_state=42)


In [ ]:
#Normalizacja - standaryzacja (Z-score)
srednia = np.mean(X_train, axis=(0,1,2), keepdims=True) # 0-epoka, 1-czestotliwosc, 2-czas <-Do usrednienia 3-kanal
std = np.std(X_train, axis=(0,1,2), keepdims=True )

print(srednia.shape)
print("Srednie dla 21 kanalow: \n")
print(srednia[0][0][0])

X_train= (X_train - srednia)/std
X_validate = (X_validate - srednia)/std
X_test = (X_test - srednia)/std

(1, 1, 1, 21)
Srednie dla 21 kanalow: 

[ 2.5577351e-10 -1.6242993e-11 -1.0967801e-10 -1.5564273e-10
 -1.2441166e-10  2.7794723e-11  2.6977293e-10  5.1325857e-11
 -6.6682165e-10 -5.0680822e-12 -4.2625233e-11 -5.1981952e-10
 -9.5994272e-12  5.6713773e-10  2.7214797e-10 -2.1874398e-10
  1.0103985e-09 -6.6682165e-10 -2.2561676e-11 -5.4360533e-10
  6.4729200e-10]


In [ ]:
model = models.Sequential()
model.add(layers.Conv2D(32, (1, 10), activation=None, padding='same', input_shape=input_shape))
  #model.add(layers.BatchNormalization())#  #
model.add(layers.Activation('relu'))
  #model.add(layers.MaxPooling2D((2,2)))#   #
model.add(layers.Conv2D(64, (3,1), activation=None, padding='same'))
  #model.add(layers.BatchNormalization())#  #
model.add(layers.Activation('relu'))
model.add(layers.MaxPooling2D((2,3)))
model.add(layers.Dropout(0.3)) # (anti-overfitting)
model.add(layers.DepthwiseConv2D(64, (3,3), activation=None, depth_multiplier=2, padding='same'))
  #model.add(layers.BatchNormalization())#  #
model.add(layers.Activation('relu'))
model.add(layers.DepthwiseConv2D(64, (2,2), activation=None, padding='same'))
  #model.add(layers.BatchNormalization())#  #
model.add(layers.Activation('relu'))
  #model.add(layers.MaxPooling2D((2,2)))#   #

model.add(layers.Flatten())
  #model.add(layers.GlobalAveragePooling2D())
model.add(layers.Dense(128, activation=None))
  #model.add(layers.BatchNormalization())
model.add(layers.Activation('relu'))
model.add(layers.Dropout(0.5)) # (anti-overfitting)
model.add(layers.Dense(l_klas, activation='softmax'))

optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001)

model.compile(optimizer = optimizer , loss = 'sparse_categorical_crossentropy' ,
              metrics = ['accuracy'],#, metrics.Precision(), metrics.Recall(), metrics.AUC()],
              jit_compile=False)
model.summary()

ReduceLROnPlateau_callback = callbacks.ReduceLROnPlateau(
    monitor='val_accuracy',
    patience = 5,
    verbose=1,
    factor=0.3,
    min_lr=0.0000001)

EarlyStopping_callback = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    start_from_epoch = 15,
    restore_best_weights=True,
    verbose=0,
    mode='min')

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 34, 421, 32)    │         6,752 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 34, 421, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 34, 421, 64)    │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 34, 421, 64)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 17, 140, 64)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 17, 140, 64)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d                │ (None, 6, 47, 128)     │       524,416 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 6, 47, 128)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d_1              │ (None, 3, 24, 128)     │       524,416 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 3, 24, 128)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 9216)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     1,179,776 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 11)             │         1,419 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,242,987 (8.56 MB)

 Trainable params: 2,242,987 (8.56 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
epochs = 70
batch_size=32  #16 # im mniejsza, tym większa dokładność, ale i więcej czasu

print("Poczatek treningu")
hist = model.fit(
    x=X_train,
    y=y_train,
    batch_size=batch_size,
    epochs=epochs,
    verbose=1,
    callbacks=[ReduceLROnPlateau_callback, EarlyStopping_callback],
    #callbacks=[ReduceLROnPlateau_callback],
    validation_data=(X_validate, y_validate),
    shuffle=True
)


Poczatek treningu
Epoch 1/70
24/24 ━━━━━━━━━━━━━━━━━━━━ 1575s 65s/step - accuracy: 0.1133 - loss: 2.3975 - val_accuracy: 0.1024 - val_loss: 2.3970 - learning_rate: 1.0000e-04
Epoch 2/70
24/24 ━━━━━━━━━━━━━━━━━━━━ 1538s 64s/step - accuracy: 0.1041 - loss: 2.3937 - val_accuracy: 0.0984 - val_loss: 2.3966 - learning_rate: 1.0000e-04
Epoch 3/70
24/24 ━━━━━━━━━━━━━━━━━━━━ 1517s 63s/step - accuracy: 0.1159 - loss: 2.3778 - val_accuracy: 0.1181 - val_loss: 2.3952 - learning_rate: 1.0000e-04
Epoch 4/70
12/24 ━━━━━━━━━━━━━━━━━━━━ 12:10 61s/step - accuracy: 0.1561 - loss: 2.3738

In [ ]:

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Skuteczność na danych testowych: {test_acc*100:.2f}%")


In [ ]:
y_predicted = model.predict(X_test)
y_p_argmax = np.argmax(y_predicted, axis=1)
#y_test_argmax = np.argmax(y_test.to_numpy(), axis=1)

print(classification_report(y_test,y_p_argmax))
matrix = confusion_matrix(y_test, y_p_argmax, labels= ["p1","r2","r3","r4","r5","r6","r7","r8","r9","r0","r1"])

disp = ConfusionMatrixDisplay(matrix)
fig, ax = plt.subplots(figsize=(12,12))
disp.plot(ax=ax)
plt.show()
plt.close()

In [ ]:
wyk, (os1,os2) = plt.subplots(1,2, figsize=(20,5))
os1.plot(hist.history['loss'],label='Strata',color='#AA0000',linewidth=2)
os1.plot(hist.history['val_loss'],label='StrataVal',color='#FFAA00',linewidth=2)
os1.set_title('Strata')
os1.set_xlabel('Epoka',fontsize=10)
os1.set_ylabel('Wartosc',fontsize=10)
os1.grid(True,linestyle='--',alpha=0.6)
os1.legend(fontsize=10)

os2.plot(hist.history['accuracy'],label='Celność',color='#00AA00',linewidth=2)
os2.plot(hist.history['val_accuracy'],label='CelnośćVal',color='#AAFF00',linewidth=2)
os2.set_title('Celnosc')
os2.set_xlabel('Epoka',fontsize=10)
os2.set_ylabel('Wartosc',fontsize=10)
os2.grid(True,linestyle='--',alpha=0.6)
os2.legend(fontsize=10)

In [ ]:
#model.save(dataSourcePath + "Najlepszy1-07-13_33" + ".keras")
#zaladowany = keras.models.load_model(dataSourcePath+nazwa+".keras")